###Step 1:
- extract chains from cdot dataset -> pubchem_cid | target | SMILES
- search the pubchem to get the interaction and pathways csv for each drug
- add them to a bigger dataset -> pubchemID | Target (list of targets + previous targets from pdb) | SMILES

In [5]:
import pandas as pd
import csv

file_path = '/home/k_ensafitakaldani001_umb_edu/BLAST/CDoT_Pf_above_70percent_18072025.csv'
df = pd.read_csv(file_path)

In [6]:
# Drop all columns but keep pubchem_cid, target, smiles
df = df[['pubchem_cid', 'target', 'smiles']]

# Remove rows where pubchem_cid is NaN
df = df.dropna(subset=['pubchem_cid'])

# Convert pubchem_cid to int
df['pubchem_cid'] = df['pubchem_cid'].astype(int)

# (Optional) If you only want the target column as a list after processing:
target_list = df['target'].tolist()


In [7]:
#interaction and pathways will give us .csv files for each pubchem id
import os
import requests

def fetch_pathways_results(pathway_ids, folder_path):
    os.makedirs(folder_path, exist_ok=True) # make the file if not there yet
    for cid in pathway_ids:
        # The url, using f-string formatting -> same for all the files
        url = f"https://pubchem.ncbi.nlm.nih.gov/sdq/sdqagent.cgi?infmt=json&outfmt=csv&query={{\"download\":\"*\",\"collection\":\"pdb\",\"order\":[\"resolution,asc\"],\"start\":1,\"limit\":10000000,\"downloadfilename\":\"pubchem_cid_{cid}_pdb\",\"where\":{{\"ands\":[{{\"cid\":\"{cid}\"}}]}}}}"
        response = requests.get(url)

        if response.status_code == 200:
            # Define the file path
            file_path = os.path.join(folder_path, f"{cid}.csv")
            # Write the content to a CSV file
            with open(file_path, 'wb') as file:
                file.write(response.content)
            print(f"Pathways data for CID {cid} has been downloaded and saved to {file_path}")
        else:
            print(f"Failed to retrieve pathways data for CID {cid}. HTTP Status Code: {response.status_code}")


In [8]:
#uncomment if you need to download the ids targets
folder_path = '/home/k_ensafitakaldani001_umb_edu/BLAST/interaction_results'
pathway_ids = df['pubchem_cid'].tolist()

fetch_pathways_results(pathway_ids , folder_path)

Pathways data for CID 9810709 has been downloaded and saved to /home/k_ensafitakaldani001_umb_edu/BLAST/interaction_results/9810709.csv
Pathways data for CID 41867 has been downloaded and saved to /home/k_ensafitakaldani001_umb_edu/BLAST/interaction_results/41867.csv
Pathways data for CID 4212 has been downloaded and saved to /home/k_ensafitakaldani001_umb_edu/BLAST/interaction_results/4212.csv
Pathways data for CID 30323 has been downloaded and saved to /home/k_ensafitakaldani001_umb_edu/BLAST/interaction_results/30323.csv
Pathways data for CID 65907 has been downloaded and saved to /home/k_ensafitakaldani001_umb_edu/BLAST/interaction_results/65907.csv
Pathways data for CID 2335 has been downloaded and saved to /home/k_ensafitakaldani001_umb_edu/BLAST/interaction_results/2335.csv
Pathways data for CID 108150 has been downloaded and saved to /home/k_ensafitakaldani001_umb_edu/BLAST/interaction_results/108150.csv
Pathways data for CID 51167 has been downloaded and saved to /home/k_ensaf

In [9]:
import os
import pandas as pd

folder_path = 'home/k_ensafitakaldani001_umb_edu/BLAST/interaction_results

empty_csvs = [] # collect all empty csv files for later
all_dfs = []  # To collect all non-empty DataFrames


for csv in os.listdir(folder_path):
    file_path = os.path.join(folder_path, csv)
        # Skip if it's not a .csv file or is a directory
    if not csv.endswith('.csv') or os.path.isdir(file_path):
        continue
    try:
        df = pd.read_csv(file_path)
        if df.empty:
            empty_csvs.append(csv)
            continue

        # Extract PubChem CID from filename (without .csv), and convert to int
        pubchem_id = int(os.path.splitext(csv)[0])
        df['pubchem_cid'] = pubchem_id

        all_dfs.append(df)

    except Exception as e:
        print(f"Error reading {csv}: {e}")
        empty_csvs.append(csv)


if all_dfs:
    combined_df = pd.concat(all_dfs, ignore_index=True)
    combined_df.to_csv(os.path.join('home/k_ensafitakaldani001_umb_edu/BLAST/,'pubchem_target.csv'), index=False)
print("Empty CSV files:", empty_csvs)


SyntaxError: unterminated string literal (detected at line 4) (1615386421.py, line 4)

### Step 2: download all the .fasta files for the proteins to do the Blast

In [2]:
import os
import pandas as pd
import requests

# Set paths
csv_path = '/home/k_ensafitakaldani001_umb_edu/BLAST/for_blast.csv'
fasta_folder = '/home/k_ensafitakaldani001_umb_edu/BLAST/fasta_files'

df = pd.read_csv(csv_path)
pdb_ids = df['pdbid'].dropna().unique()

#os.makedirs(fasta_folder, exist_ok=True)

#read df and column pdbid and download the .fasta file of each into the fasta_folder
for pdb_id in pdb_ids:
    fasta_filename = f"{pdb_id}.fasta"
    fasta_path = os.path.join(fasta_folder, fasta_filename)

    if os.path.exists(fasta_path):
        print(f"{fasta_filename} already exists.")
        continue

    url = f'https://www.rcsb.org/fasta/entry/{pdb_id}' #.fasta link on rcsb
    try:
        response = requests.get(url, timeout=10)
        if response.status_code == 200:
            with open(fasta_path, 'wb') as f:
                f.write(response.content)
            print(f" Downloaded: {fasta_filename}")
        else:
            print(f" Failed: {pdb_id} (HTTP {response.status_code})")
    except Exception as e:
        print(f" Error downloading {pdb_id}: {e}")


4I24.fasta already exists.
4I23.fasta already exists.
4FGL.fasta already exists.
 Downloaded: 7DI7.fasta
1CET.fasta already exists.
 Downloaded: 8F4Z.fasta
 Downloaded: 4V2O.fasta
 Downloaded: 5I9I.fasta
3FBV.fasta already exists.
3SDJ.fasta already exists.
 Downloaded: 6XD3.fasta
 Downloaded: 8ORM.fasta
 Downloaded: 5VC5.fasta
 Downloaded: 8FVY.fasta
 Downloaded: 8G4S.fasta
 Downloaded: 8G4I.fasta
2VRX.fasta already exists.
3ARP.fasta already exists.
3ART.fasta already exists.
1JT6.fasta already exists.
3VW0.fasta already exists.
3BT9.fasta already exists.
3BR2.fasta already exists.
3BTJ.fasta already exists.
3BR1.fasta already exists.
1OYD.fasta already exists.
3RG9.fasta already exists.
1J3K.fasta already exists.
1J3I.fasta already exists.
 Downloaded: 5EF8.fasta
 Downloaded: 9L3C.fasta
 Downloaded: 9L3S.fasta
 Downloaded: 4WNU.fasta
 Downloaded: 8R7R.fasta
 Downloaded: 6LQA.fasta
3GY3.fasta already exists.
3CR5.fasta already exists.
1D64.fasta already exists.
3HII.fasta already exi

- Clean up and add the results to a csv file such that for each .fasta name we have the hits and we put it in a csv file

- now combine the pubchem_target dataset and the new one so everything is in a one csv